---
numbering: false
---

# GEE Repeat Composite

Produce a median NDI (by default, NDVI) composite of Landsat 8/9 surface reflectance images for the same season (e.g. summer: June, July and August) in every year of a chosen range using Google Earth Engine (GEE). The composites are downloaded to a local `xarray` dataset using `xee`, and saved as one GeoTIFF per year.

In [ ]:
import ee      # The main Earth Engine library
import geemap  # A Python package for interactive mapping with Google Earth Engine
import xee     # Transfer data between Earth Engine and xarray

import pandas as pd
import rioxarray as rxr
import xarray as xr

import matplotlib.pyplot as plt

from shapely.geometry import shape

import os

Connect to Google Earth Engine. You will need to provide your own project ID (if in doubt, go to [code.earthengine.google.com](https://code.earthengine.google.com/) and see the name in the top-right corner).

In [ ]:
# =====================================
# Replace this with your own project id
my_project_name = "ee-trchudley"
# =====================================

ee.Authenticate()
ee.Initialize(project=my_project_name)

## Set parameters

Set out our search criteria. If some years return few (or zero) scenes, you may have to broaden your search criteria (e.g. a higher cloud threshold, or a longer season).

In [ ]:
# Provide a name for our area of interest (AOI)
aoi_name = "bristol"

# Provide the bounding box coordinates for the AOI in the format [min_lon, min_lat, max_lon, max_lat]
bbox = [-2.76, 51.38, -2.43, 51.53]

# Provide the range of years (inclusive) to produce composites for
year_begin = 2021
year_end = 2026

# Provide the months (inclusive) defining our season - e.g. 6 and 8 for June-August.
# month_begin must be less than or equal to month_end (i.e. seasons can't cross the new year)
month_begin = 6
month_end = 8

# Provide a cloud cover percentage threshold for filtering the search results
max_cloud_percent = 20

# Landsat collection tier: "T1" (highest geolocation quality) or "T2" (needed for some remote areas, e.g. Antarctica, with no T1 data)
tier = "T1"

# NDI to calculate
type_of_ndi = "ndvi"  # name of NDI calculated, for variable and filename purposes
r_high = "nir"        # relevant band name for r_high
r_low = "red"         # relevant band name for r_low

Visualise the bounding box to sanity check we have the correct region:

In [ ]:
aoi = ee.Geometry.BBox(*bbox)

map = geemap.Map()
map.addLayer(aoi, {}, "AOI")
map.centerObject(aoi, zoom=11)
map

## Calculate seasonal composites

Two functions: `calc_ndi()` converts a single Landsat scene to surface reflectance and calculates our NDI; `seasonal_ndi_composite()` searches for every scene within the season of a given year, calculates the NDI for each, and takes the per-pixel median. Each composite also records its date (set to the middle of the season) and the number of scenes used.

In [ ]:
def calc_ndi(image):
    """Calculate an NDI from a single Landsat 8/9 Collection 2 Level 2 scene"""
    surface_reflectance = (
        image
        .select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])  # Select the surface reflectance bands
        .multiply(0.0000275).add(-0.2)                                   # Convert DN to surface reflectance
        .rename(["blue", "green", "red", "nir", "swir1", "swir2"])
    )
    return surface_reflectance.normalizedDifference([r_high, r_low]).rename(type_of_ndi)


def seasonal_ndi_composite(year):
    """Produce a median seasonal NDI composite for a given year"""
    season_collection = (
        ee.ImageCollection(f"LANDSAT/LC09/C02/{tier}_L2")          # Landsat 9 Collection 2, Level 2 data
        .merge(ee.ImageCollection(f"LANDSAT/LC08/C02/{tier}_L2"))  # Merged with Landsat 8 Collection 2, Level 2 data
        .filterBounds(aoi)                                         # Filter to images that intersect with the AOI
        .filter(ee.Filter.calendarRange(year, year, "year"))                # Filter to the chosen year
        .filter(ee.Filter.calendarRange(month_begin, month_end, "month"))   # Filter to the chosen months
        .filter(ee.Filter.lt("CLOUD_COVER", max_cloud_percent))    # Filter to images below the cloud cover threshold
    )

    mid_month = (month_begin + month_end) // 2

    composite = (
        season_collection
        .map(calc_ndi)   # Calculate the NDI for every scene in the collection
        .median()        # Take the per-pixel median through time
        .clip(aoi)
        .set("system:time_start", ee.Date.fromYMD(year, mid_month, 15).millis())  # Record the date as mid-season
        .set("n_scenes", season_collection.size())                               # Record the number of scenes used
    )

    return composite


years = list(range(year_begin, year_end + 1))

seasonal_composites = ee.ImageCollection([seasonal_ndi_composite(year) for year in years])

Check how many scenes went into each composite. Composites built from only one or two scenes may be of poorer quality, and years with zero scenes will be empty.

In [ ]:
n_scenes = seasonal_composites.aggregate_array("n_scenes").getInfo()

pd.DataFrame({"year": years, "n_scenes": n_scenes})

## Download to `xarray` with `xee`

Define the pixel grid to download. GEE composites lose their native projection (defaulting to `EPSG:4326` at 1-degree resolution!), so we take the CRS and pixel size from an original Landsat scene, covering our AOI.

In [ ]:
# Extract the CRS and pixel size from an original Landsat scene
landsat_scene = ee.ImageCollection(f"LANDSAT/LC08/C02/{tier}_L2").filterBounds(aoi).first()
projection = landsat_scene.select("SR_B4").projection().getInfo()
crs = projection["crs"]
resolution = (projection["transform"][0], projection["transform"][4])  # (x, y) pixel size: y is negative as rows run north to south

# Fit the grid to the AOI footprint
grid = xee.helpers.fit_geometry(shape(aoi.getInfo()), grid_crs=crs, grid_scale=resolution)

print("grid parameters:", grid)

Download all composites as a single `xarray` dataset, with a `time` dimension. This is the point at which GEE actually performs the computation, so it may take a little while.

In [ ]:
ds = xr.open_dataset(
    seasonal_composites,
    engine="ee",
    **grid
)

# xarray is 'lazy' by default: this is the point at which GEE performs the computation and we download the data
ds = ds.compute()

# record the CRS, and add the number of scenes used in each composite as a coordinate
ds = ds.rio.write_crs(crs)
ds = ds.assign_coords(n_scenes=("time", n_scenes))

ds

Visualise each composite to check coverage and quality:

In [ ]:
facet = ds[type_of_ndi].plot.imshow(col="time", col_wrap=3, cmap="BrBG", vmin=-0.8, vmax=0.8, figsize=(12, 6))

for ax in facet.axes.flat:
    ax.set_aspect("equal")

plt.show()

## Save

Loop through each year and save as a GeoTIFF under a sensible name, auto-generated from parameters, within a new directory:

In [ ]:
# Construct a directory name using the AOI name and NDI type
directory = f"{aoi_name}_{type_of_ndi}_composites"

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Loop through each composite in the dataset
for i in range(ds.sizes["time"]):

    # Select the i-th composite
    ds_sel = ds[type_of_ndi].isel(time=i)

    # Get the year of the selected composite
    year = ds_sel.time.dt.year.item()

    # Construct filename using the AOI name, NDI type, year and months
    filename = f"{aoi_name}_{type_of_ndi}_{year}_months-{month_begin:02d}-{month_end:02d}_median.tif"
    filepath = os.path.join(directory, filename)

    print(f"Saving {filepath}...")

    # Save the composite as a GeoTIFF, dropping the scalar time and n_scenes coordinates
    ds_sel.drop_vars(["time", "n_scenes"]).rio.to_raster(filepath, compress="deflate")

An example of how to load the data:

In [ ]:
ds_loaded = rxr.open_rasterio(filepath, masked=True).squeeze()

ds_loaded